In [ ]:
import math
from sklearn.metrics.pairwise import euclidean_distances
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import style
import matplotlib as mpl
import scanpy as sc
import anndata
from sklearn.decomposition import NMF
import scipy
import os
import os.path as osp
import pickle as pkl
from sklearn.metrics import adjusted_rand_score

In [ ]:
# for loading DLPFC12 data
def load_DLPFC(root_dir='/Users/zhuyuling/Desktop/Dataset/DLPFC12', section_id='151507'):
    # 151507, ..., 151676
    # 12 in total, 7 layers
    # 151507: n_obs × n_vars = 4221 × 33538
    ad = sc.read_visium(path=os.path.join(root_dir, section_id), count_file=section_id+'_filtered_feature_bc_matrix.h5')
    ad.var_names_make_unique()

    gt_dir = os.path.join(root_dir, section_id, 'gt')
    gt_df = pd.read_csv(os.path.join(gt_dir, 'tissue_positions_list_GTs.txt'), sep=',', header=None, index_col=0)
    ad.obs['original_clusters'] = gt_df.loc[:, 6]
    keep_bcs = ad.obs.dropna().index
    ad = ad[keep_bcs].copy()
    ad.obs['original_clusters'] = ad.obs['original_clusters'].astype(int).astype(str)
    print("spot num", ad)
    return ad

# for loading mHypothalamus data
# already preprocessed? Xs are floats
def load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id='0.26'):
    # section id = '0.26', '0.21', '0.16', '0.11', '0.06', '0.01', '-0.04', '-0.09', '-0.14', '-0.19', '-0.24', '-0.29' 12 in total
    # cluster =     15      15      14      15      15      15      14       15       15       15      16        15
    # 9 clusters: ['BST' 'MPA' 'MPN' 'PV' 'PVH' 'PVT' 'V3' 'fx'] n_obs × n_vars = 5488 × 155
    info_file = os.path.join(root_dir, 'MERFISH_Animal1_info.xlsx')
    cnts_file = os.path.join(root_dir, 'MERFISH_Animal1_cnts.xlsx')
    xls_cnts = pd.ExcelFile(cnts_file)
    # print(xls_cnts.sheet_names)
    df_cnts = pd.read_excel(xls_cnts, section_id)

    xls_info = pd.ExcelFile(info_file)
    df_info = pd.read_excel(xls_info, section_id)
    # print(df_cnts, df_info)
    spatial_X = df_info.to_numpy()
    obs_ = df_info
    if len(df_info.columns) == 5:
        obs_.columns = ['psuedo_barcodes', 'x', 'y', 'original_clusters', 'Neuron_cluster_ID']
    elif len(df_info.columns) == 6:
        obs_.columns = ['psuedo_barcodes', 'x', 'y', 'cell_types', 'Neuron_cluster_ID', 'original_clusters']
    obs_.index = obs_['psuedo_barcodes'].tolist()
    # print(obs_)

    var_ = df_cnts.iloc[:, 0]
    var_ = pd.DataFrame(var_)
    # print(var_)

    ad = anndata.AnnData(X=df_cnts.iloc[:,1:].T, obs=obs_, var=var_)
    spatial = np.vstack((ad.obs['x'].to_numpy(), ad.obs['y'].to_numpy()))
    ad.obsm['spatial'] = spatial.T
    return ad

In [ ]:
 
section_ids_list= ['-0.09', '-0.14', '-0.19', '-0.24']
data_1 = load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id='-0.04')
meta_data_1=data_1.obsm['spatial'] 
median_1=np.median(meta_data_1,axis=0)
 
result=pd.DataFrame(meta_data_1)
result=result.set_index(data_1.obs_names)
result = result.rename(columns={0: 'X', 1: 'Y'})
result['original_clusters']=data_1.obs['original_clusters']
result['slice']='-0.04'
#result['X']=data_1.obs_names
ds=pd.DataFrame()
ds = pd.concat([ds,result],axis=0)
for index,s in enumerate(section_ids_list):
    data_2= load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id=s)
    meta_data_2=data_2.obsm['spatial'] 
    
    median_2=np.median(meta_data_2,axis=0)
    result=meta_data_2-median_2+median_1
    result=pd.DataFrame(result)
    result=result.set_index(data_2.obs_names)
    result = result.rename(columns={0: 'X', 1: 'Y'})
    result['original_clusters']=data_2.obs['original_clusters']
    result['slice']=s
    #result['X']=data_2.obs_names
    ds = pd.concat([ds,result],axis=0)
ds.to_csv('/Users/zhuyuling/Desktop/Dataset/alignment/mHy/coord.csv')
 

In [ ]:
 
df_1=pd.read_csv('/Users/zhuyuling/Desktop/Dataset/alignment/mHy/coord.csv')
 
section_ids_list=['-0.04','-0.09', '-0.14', '-0.19', '-0.24']
 
for index in range(len(section_ids_list)-1):
    slice_1= df_1.loc[df_1['slice']==float(section_ids_list[index]),['X','Y']]
    slice_2=df_1.loc[df_1['slice']==float(section_ids_list[index+1]),['X','Y']]
    alignment=euclidean_distances(slice_1,slice_2)
    alignment = np.exp(-alignment)
    dataset=section_ids_list[index]+'_'+str(section_ids_list[index+1])
    map_path=osp.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', dataset, dataset+'_AlignmentPi.pkl')
    if not os.path.exists(osp.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', dataset)):
        os.makedirs(osp.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', dataset))
    with open(map_path,'wb') as f:
        pkl.dump(alignment, f)

In [ ]:
def cal_layer_based_alignment_result_mHypothalamus(alignment, s1, s2):
    #fei added
    labels = []
    labels.extend(s1.obs['original_clusters'])
    labels.extend(s2.obs['original_clusters'])

    res = []
    #l_dict = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5": 5, "6": 6}
    cnt0 = 0
    ir = 0
    max_smallerThan0 = []
    sum_smallerThan0 = []
    min_smallerThan0 = []
    
    for i, elem in enumerate(alignment):
        #if np.sum(elem)>0:
        if np.max(elem) - np.min(elem) != 0:
            if labels[i] == labels[elem.argmax() + alignment.shape[0]]:
                cnt0 += 1
             
        else:
            ir += 1
            max_smallerThan0.append(np.max(elem))
            sum_smallerThan0.append(np.sum(elem))
            min_smallerThan0.append(np.min(elem))
    print("the number of sum alignment equals 0:", str(ir))
    if len(sum_smallerThan0)>0:
        print("sum in sum alignment equals 0 is between", np.min(sum_smallerThan0), np.max(sum_smallerThan0))
        print("max in sum alignment equals 0 is between", np.min(max_smallerThan0), np.max(max_smallerThan0))
        print("min in sum alignment equals 0 is between", np.min(min_smallerThan0), np.max(min_smallerThan0))
    #print(alignment.shape[0])
    #print(cnt0/alignment.shape[0], cnt1/alignment.shape[0], cnt2/alignment.shape[0], cnt3/alignment.shape[0], cnt4/alignment.shape[0], cnt5/alignment.shape[0], cnt6/alignment.shape[0])
    #res.extend([cnt0/alignment.shape[0], cnt1/alignment.shape[0], cnt2/alignment.shape[0], cnt3/alignment.shape[0], cnt4/alignment.shape[0], cnt5/alignment.shape[0], cnt6/alignment.shape[0]])
    #res.extend([cnt0, cnt1, cnt2, cnt3, cnt4, cnt5, cnt6])
    res.extend([cnt0])
    return res

def cal_paste_paste2_lba(dataset, s1, s2):
    map_path=osp.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', dataset, dataset+'_AlignmentPi.pkl')
    #alignment=np.load(osp.join(input, method_path, file_name+"_"+method_path+"_AlignmentPi.npy"))
    #map_path = osp.join(input, method_path, file_name +'_'+dataset+'_AlignmentPi.pkl')
    with open(map_path, 'rb') as f:
        ali_o = pkl.load(f)
    
    lba = cal_layer_based_alignment_result_mHypothalamus(ali_o, s1, s2)
    sub_lba = [element/s1.shape[0] for element in lba]  
    return sub_lba
 

slices = ['-0.04','-0.09', '-0.14', '-0.19', '-0.24']
#df_1=pd.read_csv('/Users/zhuyuling/Desktop/Dataset/alignment/DLPFC4/151669_151672_coord.csv')
 
df=pd.DataFrame()
for index in range(0,len(slices)-1):
    slice1= load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id=slices[index])
    slice2= load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id=slices[index+1])
    dataset=str(slices[index])+'_'+str(slices[index+1])
    laa=cal_paste_paste2_lba(dataset,slice1,slice2)
    df2=pd.DataFrame({
        dataset:laa
        })
    df = pd.concat([df,df2],ignore_index=False,axis=1)
 
df.to_csv('/Users/zhuyuling/Desktop/Dataset/alignment/mHy/mHypothalamus_LAA.csv')

In [ ]:
def compute_alignment_ari(sliceA, sliceB, pi):
    mapped_clusters = []
    for j in range(pi.shape[1]):
        mapping = pi[:, j]
        #original: if np.sum(mapping) > 0:
        if np.max(mapping) - np.min(mapping) != 0:
            i = np.argmax(mapping)
            mapped_clusters.append(sliceA.obs['original_clusters'][i])
        else:
            mapped_clusters.append("NULL")
    assert len(sliceB.obs['original_clusters']) == len(mapped_clusters)
    true_clusters_mapped_region = []
    mapped_clusters_mapped_region = []
    for i in range(len(sliceB.obs['original_clusters'])):
        if mapped_clusters[i] != "NULL":
            true_clusters_mapped_region.append(sliceB.obs['original_clusters'][i])
            mapped_clusters_mapped_region.append(mapped_clusters[i])

    ari = adjusted_rand_score(true_clusters_mapped_region, mapped_clusters_mapped_region)
    return ari

def cal_paste_paste2_ari(dataset,s1,s2):
    sub_ari=[]
        #alignment=np.load(osp.join(input, method_path, file_name+"_"+method_path+"_AlignmentPi.npy"))
    map_path=osp.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', dataset, dataset+'_AlignmentPi.pkl')
    with open(map_path, 'rb') as f:
        ali_o = pkl.load(f)
         
    arI = compute_alignment_ari(s1, s2, ali_o)
    sub_ari.append(arI)
    return sub_ari
slices = ['-0.04','-0.09', '-0.14', '-0.19', '-0.24']
 
df=pd.DataFrame()
for index in range(0,len(slices)-1):
    slice1= load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id=slices[index])
    slice2= load_mHypothalamus(root_dir='/Users/zhuyuling/Desktop/Dataset/mHypothalamus', section_id=slices[index+1])
     
    dataset=slices[index]+'_'+str(slices[index+1])
    ari=cal_paste_paste2_ari(dataset,slice1,slice2)
    df2=pd.DataFrame({
        dataset:ari
        })
    df = pd.concat([df,df2],ignore_index=False,axis=1)

df.to_csv('/Users/zhuyuling/Desktop/Dataset/alignment/mHy/mHypothalamus_ARI.csv')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from SPACEL import Scube
import os
cmap = []
for e in plt.get_cmap('tab10').colors:
    cmap.append(matplotlib.colors.rgb2hex(e))
coo = pd.DataFrame()
slices = ['-0.04','-0.09', '-0.14', '-0.19', '-0.24']
df_1=pd.read_csv('/Users/zhuyuling/Desktop/Dataset/alignment/mHy/coord.csv',index_col=0)
print(cmap)
for index, i in enumerate(slices):
    embedding=df_1.loc[df_1['slice'] == float(i),]
    embedding['Z'] = index
    embedding['celltype_colors'] = embedding['original_clusters'].replace(dict(zip(['BST', 'MPA', 'MPN', 'PV', 'PVH', 'PVT', 'V3', 'fx'],cmap)))
    
    coo = pd.concat([coo,embedding],axis=0)
 
coo = coo[['X', 'Y','Z','original_clusters','celltype_colors','slice']]
coords=coo.values[:,:3]
'''
coo['X'], coo['Z'] = coo['Z'], coo['X']
coo['X'], coo['Y'] = coo['Y'], coo['X']
coo['Z'] = -coo['Z']
print(coords)
#coords = coo.values[:,:3]
'''
coo.to_csv('/Users/zhuyuling/Desktop/Dataset/alignment/Scube_outputs/scHolography_mHypothalamus_aligned_coordinates_colors.csv')
'''
coords[:,[2, 0]] = coords[:,[0, 2]]
coords[:,[1, 0]] = coords[:,[0, 1]]
coords[:, 2] = -coords[:, 2]
'''
#2. save fig
fig = Scube.plot_3d(coords, val=None, color=coo['celltype_colors'], elev=20, azim=110, frameon=False, show=True, s=35, return_fig=True) # Scube is from SPACEL

fig.savefig(os.path.join('/Users/zhuyuling/Desktop/Dataset/alignment/mHy', "scHolography_gtAndScube_mHypothalamus_3dviz_angle20_110_1128_24_s=35.pdf"), bbox_inches="tight")
